In [ ]:
# ✅ ADIM 1: Google Drive'ı Bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ ADIM 2: Drive'daki ZIP'i aç
# NOT: colon_image_sets.zip dosyasını Google Drive ana dizinine yüklemiş olmalısın
import zipfile

zip_path = '/content/drive/MyDrive/colon_image_sets.zip'
extract_path = 'colon_dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('ZIP basariyla acildi!')

In [ ]:
## ==========================================
# ✅ ADIM 3: EĞİTİM - COLON CANCER CLASSIFICATION
# Model: EfficientNetB3 | Epoch: 15 | batch_size: 32
# Oranlar: %70-%30 (Test) | %80-%20 (Train-Val)
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras import regularizers
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')

# 1. VERİ SETİ YOLLARINI HAZIRLA
data_dir = 'colon_dataset/colon_image_sets'
filepaths = []
labels = []

if not os.path.exists(data_dir):
    print(f"HATA: '{data_dir}' klasoru bulunamadi!")
else:
    for sub_folder in os.listdir(data_dir):
        sub_folder_path = os.path.join(data_dir, sub_folder)
        if os.path.isdir(sub_folder_path):
            for file in os.listdir(sub_folder_path):
                if file.endswith(('.jpg', '.jpeg', '.png')):
                    filepaths.append(os.path.join(sub_folder_path, file))
                    labels.append(sub_folder)

    df = pd.DataFrame({'filepaths': filepaths, 'labels': labels})
    print(f'Toplam resim sayisi: {len(df)}')

    # -- ORANLAR: %70 Train+Val / %30 Test -- 
    train_temp_df, test_df = train_test_split(df, train_size=0.7, shuffle=True, random_state=123)
    train_df, valid_df = train_test_split(train_temp_df, train_size=0.8, shuffle=True, random_state=123)

    print(f'Egitim (Train): {len(train_df)}')
    print(f'Dogrulama (Valid): {len(valid_df)}')
    print(f'Test (Test): {len(test_df)}')

    img_size = (224, 224)
    batch_size = 32

    tr_gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    ts_gen = ImageDataGenerator(rescale=1./255)

    train_gen = tr_gen.flow_from_dataframe(train_df, x_col='filepaths', y_col='labels',
                                           target_size=img_size, class_mode='categorical', batch_size=batch_size)
    valid_gen = ts_gen.flow_from_dataframe(valid_df, x_col='filepaths', y_col='labels',
                                           target_size=img_size, class_mode='categorical', batch_size=batch_size)
    test_gen = ts_gen.flow_from_dataframe(test_df, x_col='filepaths', y_col='labels',
                                          target_size=img_size, class_mode='categorical', batch_size=batch_size, shuffle=False)

    classes = list(train_gen.class_indices.keys())

    # MODEL: EfficientNetB3
    base_model = tf.keras.applications.EfficientNetB3(include_top=False, weights='imagenet', input_shape=(224,224,3))

    model = Sequential([
        base_model,
        BatchNormalization(),
        GlobalAveragePooling2D(),
        Dense(512, kernel_regularizer=regularizers.l2(0.01), activation='relu'),
        Dropout(0.5),
        Dense(len(classes), activation='softmax')
    ])

    model.compile(tf.keras.optimizers.Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

    epochs = 15
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
    ]

    print('\n--- EGITIM BASLIYOR ---')
    history = model.fit(train_gen, epochs=epochs, validation_data=valid_gen, callbacks=callbacks)

    # GRAFİKLER
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Egitim')
    plt.plot(history.history['val_accuracy'], label='Dogrulama')
    plt.title('Dogruluk'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Egitim')
    plt.plot(history.history['val_loss'], label='Dogrulama')
    plt.title('Kayip'); plt.legend()
    plt.show()

    test_loss, test_acc = model.evaluate(test_gen)
    print(f'\nFinal Test Dogrulugu: %{test_acc*100:.2f}')

    # TAHMİN GÖRSELLEŞTİRME - Teşhis Odaklı Renkler
    test_gen.reset()
    test_images, test_labels_batch = next(test_gen)
    predictions = model.predict(test_images)

    plt.figure(figsize=(16, 16))
    for i in range(min(16, len(test_images))):
        ax = plt.subplot(4, 4, i + 1)
        img = (test_images[i] * 255).astype(np.uint8)
        plt.imshow(img)

        true_idx = np.argmax(test_labels_batch[i])
        pred_idx = np.argmax(predictions[i])
        true_label = classes[true_idx]
        pred_label = classes[pred_idx]

        is_aca_pred = 'aca' in pred_label.lower()
        is_aca_true = 'aca' in true_label.lower()

        pred_type = 'Kotu Huylu' if is_aca_pred else 'Iyi Huylu'
        true_type = 'Kotu Huylu' if is_aca_true else 'Iyi Huylu'

        color = 'red' if is_aca_pred else 'green'
        status = 'DOGRU' if true_idx == pred_idx else 'YANLIS'

        title_text = f'Tahmin: {pred_label} ({pred_type})\nGercek: {true_label} ({true_type})\nDurum: {status}'

        plt.title(title_text, color=color, fontsize=10, fontweight='bold')

        # KARE CERCEVE
        rect = plt.Rectangle((0, 0), img.shape[1], img.shape[0],
                              linewidth=10, edgecolor=color, facecolor='none',
                              transform=ax.transData)
        ax.add_patch(rect)
        plt.axis('off')

    plt.suptitle('Test Tahminleri - Kirmizi: Kotu Huylu (colon_aca) | Yesil: Iyi Huylu (colon_n)', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ✅ ADIM 4: DOGRULUK TABLOSU (Confusion Matrix)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix

# Tüm test seti üzerinde tahmin
test_gen.reset()
all_preds = model.predict(test_gen)
y_pred = np.argmax(all_preds, axis=1)
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

# Renk matrisi: köşegen=yeşil, diğerleri=kırmızı
fig, ax = plt.subplots(figsize=(7, 6))

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i == j:
            color = '#2ecc71' if cm[i, j] > 0 else '#27ae60'
        else:
            color = '#e74c3c' if cm[i, j] > 0 else '#2c3e50'
        rect = mpatches.FancyBboxPatch(
            (j - 0.45, i - 0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            linewidth=2, edgecolor='white',
            facecolor=color, alpha=0.85
        )
        ax.add_patch(rect)
        ax.text(j, i, str(cm[i, j]),
                ha='center', va='center',
                fontsize=22, fontweight='bold', color='white')

ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, fontsize=13, fontweight='bold')
ax.set_yticklabels(class_names, fontsize=13, fontweight='bold')
ax.set_xlabel('Tahmin Edilen', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Gercek Sinif', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Dogruluk Tablosu (Confusion Matrix)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlim(-0.5, len(class_names) - 0.5)
ax.set_ylim(len(class_names) - 0.5, -0.5)
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#16213e')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')

# Legend
green_patch = mpatches.Patch(color='#2ecc71', label='Dogru Tahmin')
red_patch   = mpatches.Patch(color='#e74c3c', label='Yanlis Tahmin')
ax.legend(handles=[green_patch, red_patch], loc='upper right',
          fontsize=11, framealpha=0.3, labelcolor='white',
          facecolor='#16213e', edgecolor='white')

plt.tight_layout()
plt.show()

# Özet istatistikler
total = cm.sum()
correct = np.trace(cm)
print(f'Toplam Test Resmi : {total}')
print(f'Dogru Tahmin      : {correct}  (%{correct/total*100:.2f})')
print(f'Yanlis Tahmin     : {total-correct}  (%{(total-correct)/total*100:.2f})')